# Test Embedding-Vector Similarity

Folgend soll die Berechung der Ähnlichkeit von Embedding-Vektoren getestet werden. Für die Erzeugung der Text-Embeddings werden die Modelle "nvidia/llama-nemotron-embed-1b-v2" und "intfloat/multilingual-e5-large" getestet, die Ergebnisse verglichen und abschließend ein Modell für den Einsatz in den Experimenten gewählt.

Für die Berechnung der Ähnlichkeit werden die Methoden Cosine-Similarity, Euclidean-Distance (L2 Distance), Dot-Product und City-Block Distance/Manhattan Distance angewendet.

Anhand einer durch ein LLM (Gemini 3 3.5 Flash) formulierten Beispiel-Aussage und einer dazu ähnlichen sowie gegensätzlichen Aussage wird die Ähnlichkeit mittels der zuvor genannten Methoden berechnet. Anschließend werden die Ergebnisse verglichen, die Vor- und Nachteile aufgeführt und eine Methode zur Berechnung der Ähnlichkeiten für die weitere Verwendung ausgewählt.

Zudem wird eine Prüfung der Korrelation der erzielten Ergebnisse und deren Ausrichtung gegenüber den durch Annotatoren bewerteten Ähnlichkeiten von Satz-Paaren mit Hilfe des SemRel2024-Datensatzes durchgefüht.

## Setup

### Pakete laden

In [1]:
import config

import time
import math

import numpy as np

from openai import OpenAI

# Hugging-Face datasets
import datasets

import scipy.stats as stats

### Variablen+Funktionen

In [2]:
# Embedding-Modell
model_embedding = 'nvidia/llama-nemotron-embed-1b-v2'
#model_embedding = 'intfloat/multilingual-e5-large'

# Embedding OpenAI-Client
openai_client_embedding = OpenAI(
    base_url=config.openai_client['embedding']['base_url'],
    api_key=config.openai_client['api_key']
)

## Methoden zur Berechnung der Ähnlichkeit

- Embedding-Vektoren für alle drei Aussagen erzeugen
- Ähnlichkeit der Embedding-Vektoren für erste Aussage gegenüber ähnlicher und gegensätzlicher Aussage berechnen

**Cosine-Similarity**

Die Cosine-Similarity (Kosinus-Ähnlichkeit) misst den Kosinus des Winkels zwischen zwei Vektoren. Für zwei Texte $t_1$ und $t_2$ wird die Ähnlichkeit ihrer $n$-dimensionalen Embedding-Vektoren $\mathbf{v}_1 = (v_{11}, v_{12}, \dots, v_{1n})$ und $\mathbf{v}_2 = (v_{21}, v_{22}, \dots, v_{2n})$ wie folgt berechnet:

$$\mathbf{v}_1 = Emb(t_1) \in \mathbb{R}^n$$

$$\mathbf{v}_2 = Emb(t_2) \in \mathbb{R}^n$$

$$\text{Sim}(t_1, t_2) = \cos(\theta) = \frac{\mathbf{v}_1 \cdot \mathbf{v}_2}{\|\mathbf{v}_1\| \|\mathbf{v}_2\|}$$

Der resultierende Ähnlichkeitswert reicht von -1 bis 1. Ein Wert von 1 bedeutet, dass die Vektoren in genau dieselbe Richtung zeigen (identische Bedeutung). 0 bedeutet, dass sie senkrecht zueinander stehen (kein Zusammenhang). -1 bedeutet, dass die Vektoren gegensätzlich verlaufen.

Die Methode misst den Winkel zwischen zwei Vektoren im Raum, unabhängig von deren Länge (Magnitude). Längere Texte enthalten mehr Wörter, was die Vektoren "länger" macht, aber nicht zwingend deren thematische Richtung ändert. Der Vergleich mittels Cosine-Similarity ist daher bei (stark) variierenden Textlängen besser geeignet. Die Berechnung ist durch die Normierung im Hintergrund etwas aufwendiger als ein reines Skalarprodukt.

In [3]:
# Cosine similarity
def cosine_similarity(a, b):
    return float(np.dot(a, b)) / (float(np.linalg.norm(a)) * float(np.linalg.norm(b)))

**Dot-Product**

Das Dot-Product (dt. Skalarprodukt) zweier $n$-dimensionaler Vektoren $\mathbf{a}=(a_1, a_2, \dots, a_n)$ und $\mathbf{b}=(b_1, b_2, \dots, b_n)$ wird aus der Summe ihrer entsprechenden komponentenweisen Produkte berechnet:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = a_1 b_1 + a_2 b_2 + \dots + a_n b_n$$

Das Skalarprodukt steht in direktem Zusammenhang mit dem Kosinus des Winkels $\theta$ zwischen den beiden Vektoren:

$$\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\| \|\mathbf{b}\| \cos(\theta)$$

Wenn die Vektoren normiert sind (Länge 1 haben), entspricht das Dot-Product dem Kosinus des Winkels (siehe Cosine-Similarity).

Die Methode berücksichtigt sowohl den Winkel als auch die Länge der Vektoren. Wenn die Vektoren nicht normiert sind, werden längere Texte oder häufiger vorkommende Begriffe bevorzugt, da sie die Vektoren künstlich vergrößern. Die Methode ist extrem schnell zu berechnen (hocheffizient auf GPUs) und geeignet, wenn die Vektoren bereits normiert sind (Länge=1).

In [4]:
# Dot Product
def dot_product(a, b):
    return np.dot(a, b)

**Euclidean distance**

Die euklidische Distanz (auch als $L^2$ norm or $L^2$ distance bezeichnet) misst den geradlinigen Abstand zwischen zwei Punkten (oder Vektoren) in einem hochdimensionalen euklidischen Raum. Die Distanz wird berechnet, indem die quadrierten Unterschiede entlang jeder einzelnen Dimension summiert werden. Für zwei $n$-dimensionale Vektoren $\mathbf{a} = (a_1, a_2, \dots, a_n)$ und $\mathbf{b} = (b_1, b_2, \dots, b_n)$ wird die euklidische Distanz wie folgt berechnet:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

Das Ergebnis ist ein einzelner, skalarer Wert $\ge 0$. Je geringer die Distanz zwischen beiden Vektoren ist, um so ähnlicher sind sich diese. Wenn die Distanz exakt Null ist, sind die beiden Vektoren identisch ($\mathbf{a} = \mathbf{b}$).

Die Methode ist geeignet, wenn die absolute Position im Raum wichtig ist und die Vektoren exakt dieselbe Länge haben (normiert sind). Allerdings ist sie bei stark unterschiedlichen Textlängen nicht robust. Ein kurzer und ein langer Text mit dem selben Inhalt haben eine große euklidische Distanz, obwohl sie semantisch identisch sind.

In [5]:
# Euclidean distance
def euclidean_distance(a, b):
    #return math.dist(a, b)
    #return np.sqrt(((np.array(a) - np.array(b))**2).sum())
    return np.linalg.norm(np.array(a) - np.array(b))

**City-Block Distance/Manhattan Distance**

Die City-Block Distance (auch als Manhattan-Distance oder $L_1$-Distance bezeichnet) misst den geradlinigen Abstand zwischen zwei Punkten (oder Vektoren) in einem hochdimensionalen euklidischen Raum. Die Distanz wird berechnet, indem die absoluten Unterschiede entlang jeder einzelnen Dimension summiert werden, ähnlich der euklidischen Distanz. Für zwei $n$-dimensionale Vektoren $\mathbf{a} = (a_1, a_2, \dots, a_n)$ und $\mathbf{b} = (b_1, b_2, \dots, b_n)$ wird die City Block Distance wie folgt berechnet:

$$d_{Cityblock}(\mathbf{a}, \mathbf{b}) = \sum_{i=1}^{n} |a_i - b_i|$$

Wie für die Euclidean-Distance ist das Ergebnis ist ein einzelner, skalarer Wert $\ge 0$. Da es sich auch hierbei um eine Distanz-Maß handelt, bedeutet ein kleinerer Wert eine höhere Ähnlichkeit.

Der Einsatz der Methode ist beim Vorhandensein hochdimensionaler Räume geeignet, in denen die euklidische Distanz an Aussagekraft verlieren kann bzw. die Interpretierbarkeit der Ergebnisse nicht möglich ist. Gegenüber einzelnen Ausreißern ist sie in bestimmten Dimensionen zudem robuster als die euklidische Distanz und ignoriert ebenso die Richtungsähnlichkeit und ist anfällig bei stark unterschiedlichen Textlängen.

In [6]:
# City Block Distance/Manhattan Distance
def city_block(a, b):
    #return np.sum(np.abs(np.array(a) - np.array(b)))
    return np.abs((np.array(a) - np.array(b))).sum()

## SemRel2024 Dataset

https://huggingface.co/datasets/SemRel/SemRel2024

Datensatz mit Satz-Paaren und Score-Wert des Maßes der Beziehung zwischen diesen zur Evaluierung der Semantic relatedness.

Der Score-Wert liegt zwischen 0 (vollständig ohne Bezug) und 1 (maximaler Bezug).

In [7]:
dataset_semRel2024 = datasets.load_dataset("SemRel/SemRel2024", "eng")

In [8]:
#sample = 500

# Dataset splitten, random sample für Test selektieren
#dataset_semRel2024_split = dataset_semRel2024['test'].train_test_split(test_size=sample / dataset_semRel2024['test'].shape[0], seed=42)

dataset_semRel2024_split = dataset_semRel2024['test']

### Stats

Statistiken zum Datensatz

In [9]:
dataset_semRel2024_split_test_label = np.array(dataset_semRel2024_split['label'])

print(f'mean: {dataset_semRel2024_split_test_label.mean()}')
print(f'min: {dataset_semRel2024_split_test_label.min()}')
print(f'max: {dataset_semRel2024_split_test_label.max()}')
print(f'median: {np.median(dataset_semRel2024_split_test_label)}')
print(f'std: {dataset_semRel2024_split_test_label.std()}')
print(f'var: {dataset_semRel2024_split_test_label.var()}')

mean: 0.5166153846153846
min: 0.12
max: 0.97
median: 0.48
std: 0.16995928854673878
var: 0.028886159763313612


### Vector-Similarity berechnen

Embedding-Vektoren für Satz-Paare erzeugen und Ähnlichkeiten (Cosine-Similarity, Euclidean-Distance, Dot-Product und City-Block) berechnen.

In [ ]:
# Zeitstempel
time_start = time.time()

# Embeddings für sentence1 erzeugen
embeddings_sentence1 = openai_client_embedding.embeddings.create(
    input=list(dataset_semRel2024_split['sentence1']),
    model=model_embedding
)

# Embeddings für sentence1 erzeugen
embeddings_sentence2 = openai_client_embedding.embeddings.create(
    input=list(dataset_semRel2024_split['sentence2']),
    model=model_embedding
)

# Dictionary für Werte
embeddings_scores = {'cosine_similarity': [], 'euclidean_distance': [], 'dot_product': [], 'city_block': []}

# Über alle Embeddings iterieren
for i in range(len(embeddings_sentence1.data)):
    embeddings_scores['dot_product'].append(dot_product(embeddings_sentence1.data[i].embedding, embeddings_sentence2.data[i].embedding))

print(time.time() - time_start)

In [11]:
# Embeddings für sentence1 erzeugen
embeddings_sentence1 = openai_client_embedding.embeddings.create(
    input=list(dataset_semRel2024_split['sentence1']),
    model=model_embedding
)

# Embeddings für sentence1 erzeugen
embeddings_sentence2 = openai_client_embedding.embeddings.create(
    input=list(dataset_semRel2024_split['sentence2']),
    model=model_embedding
)

# Dictionary für Werte
embeddings_scores = {'cosine_similarity': [], 'euclidean_distance': [], 'dot_product': [], 'city_block': []}

# Über alle Embeddings iterieren
for i in range(len(embeddings_sentence1.data)):
    embeddings_scores['cosine_similarity'].append(cosine_similarity(embeddings_sentence1.data[i].embedding, embeddings_sentence2.data[i].embedding))
    embeddings_scores['euclidean_distance'].append(euclidean_distance(embeddings_sentence1.data[i].embedding, embeddings_sentence2.data[i].embedding))
    embeddings_scores['dot_product'].append(dot_product(embeddings_sentence1.data[i].embedding, embeddings_sentence2.data[i].embedding))
    embeddings_scores['city_block'].append(city_block(embeddings_sentence1.data[i].embedding, embeddings_sentence2.data[i].embedding))

In [12]:
embeddings_scores['cosine_similarity'] = np.array(embeddings_scores['cosine_similarity'])
embeddings_scores['euclidean_distance'] = np.array(embeddings_scores['euclidean_distance'])
embeddings_scores['dot_product'] = np.array(embeddings_scores['dot_product'])
embeddings_scores['city_block'] = np.array(embeddings_scores['city_block'])

In [13]:
print(f'mean: {embeddings_scores["cosine_similarity"].mean()}')
print(f'min: {embeddings_scores["cosine_similarity"].min()}')
print(f'max: {embeddings_scores["cosine_similarity"].max()}')
print(f'median: {np.median(embeddings_scores["cosine_similarity"])}')
print(f'std: {embeddings_scores["cosine_similarity"].std()}')
print(f'var: {embeddings_scores["cosine_similarity"].var()}')

mean: 0.48216294590959985
min: -0.072895027330577
max: 0.9830247407597856
median: 0.43043070070260414
std: 0.2665649216648974
var: 0.07105685746221291


Modell: nvidia/llama-nemotron-embed-1b-v2

mean: 0.4821407447734243  
min: -0.0730990283079947  
max: 0.9830247412715899  
median: 0.4304306992026178  
std: 0.2665749589494724  
var: 0.07106220873891289

Modell: intfloat/multilingual-e5-large

mean: 0.8500728392155485  
min: 0.6569694570670067  
max: 0.9955085366537195  
median: 0.8314041446719517  
std: 0.07775102797158873  
var: 0.0060452223506387735

In [14]:
print(f'mean: {embeddings_scores["dot_product"].mean()}')
print(f'min: {embeddings_scores["dot_product"].min()}')
print(f'max: {embeddings_scores["dot_product"].max()}')
print(f'median: {np.median(embeddings_scores["dot_product"])}')
print(f'std: {embeddings_scores["dot_product"].std()}')
print(f'var: {embeddings_scores["dot_product"].var()}')

mean: 0.48216294596000175
min: -0.07289502622874072
max: 0.9830247088620383
median: 0.43043071291737145
std: 0.26656492133764803
var: 0.0710568572877465


Modell: nvidia/llama-nemotron-embed-1b-v2

mean: 0.4821407448511662  
min: -0.0730990313018568  
max: 0.9830247077086786  
median: 0.43043070956938073  
std: 0.2665749586507046  
var: 0.07106220857962486

Modell: intfloat/multilingual-e5-large

mean: 0.8500728397012073  
min: 0.6569694669517149  
max: 0.9955085440140676  
median: 0.8314041325665822  
std: 0.0777510282504  
var: 0.0060452223939945

In [15]:
print(f'mean: {embeddings_scores["euclidean_distance"].mean()}')
print(f'min: {embeddings_scores["euclidean_distance"].min()}')
print(f'max: {embeddings_scores["euclidean_distance"].max()}')
print(f'median: {np.median(embeddings_scores["euclidean_distance"])}')
print(f'std: {embeddings_scores["euclidean_distance"].std()}')
print(f'var: {embeddings_scores["euclidean_distance"].var()}')

mean: 0.9732915649421525
min: 0.1842566616944539
max: 1.4648515358993615
median: 1.067304371676612
std: 0.2972837676256637
var: 0.08837763849370962


Modell: nvidia/llama-nemotron-embed-1b-v2

mean: 0.9733112951254812  
min: 0.18425665876072533  
max: 1.464990834277203  
median: 1.0673043707893628  
std: 0.29729385050386736  
var: 0.08838363354741582

Modell: intfloat/multilingual-e5-large

mean: 0.5236792025119907  
min: 0.09477830320795848  
max: 0.8282880514581903  
median: 0.5806821003749957  
std: 0.16004503908626388  
var: 0.025614414536123734

In [16]:
print(f'mean: {embeddings_scores["city_block"].mean()}')
print(f'min: {embeddings_scores["city_block"].min()}')
print(f'max: {embeddings_scores["city_block"].max()}')
print(f'median: {np.median(embeddings_scores["city_block"])}')
print(f'std: {embeddings_scores["city_block"].std()}')
print(f'var: {embeddings_scores["city_block"].var()}')

mean: 34.97284147870268
min: 6.602603124106736
max: 52.57618307116172
median: 38.34517421844305
std: 10.681600887991525
var: 114.09659753034133


Modell: nvidia/llama-nemotron-embed-1b-v2

mean: 34.9734878913591  
min: 6.602602974163801  
max: 52.57815871076127  
median: 38.334592210074334  
std: 10.682008624822318  
var: 114.10530826077837

Modell: intfloat/multilingual-e5-large

mean: 13.312634550989001  
min: 2.433179124702292  
max: 20.961504897753002  
median: 14.748577949264927  
std: 4.06685063640353  
var: 16.5392740988158

Modell: nvidia/llama-nemotron-embed-1b-v2

| Score | mean | min | max | median | std | var |
| - | - | - | - | - | - | - |
| **Referenz** | 0.5166153846153846 | 0.12 | 0.97 | 0.48 | 0.16995928854673878 | 0.028886159763313612 |
| **Cosine-Similarity** | 0.4821407447734243 | -0.0730990283079947 |  0.9830247412715899 | 0.4304306992026178 | 0.2665749589494724 | 0.07106220873891289 |
| **Dot-Product** | 0.4821407448511662 | -0.0730990313018568 | 0.9830247077086786 | 0.43043070956938073 | 0.2665749586507046 | 0.07106220857962486 |
| **Euclidean-Distance** | 0.9733112951254812 | 0.18425665876072533 | 1.464990834277203 | 1.0673043707893628 | 0.29729385050386736 | 0.08838363354741582 |
| **City-Block** | 34.9734878913591 | 6.602602974163801 | 52.57815871076127 | 38.334592210074334 | 10.682008624822318 | 114.10530826077837 |

Modell: intfloat/multilingual-e5-large

| Score | mean | min | max | median | std | var |
| - | - | - | - | - | - | - |
| **Referenz** | 0.5166153846153846 | 0.12 | 0.97 | 0.48 | 0.16995928854673878 | 0.028886159763313612 |
| **Cosine-Similarity** | 0.8500728392155485 | 0.6569694570670067 |  0.9955085366537195 | 0.8314041446719517 | 0.07775102797158873 | 0.0060452223506387735 |
| **Dot-Product** | 0.8500728397012073 | 0.6569694669517149 | 0.9955085440140676 | 0.8314041325665822 | 0.0777510282504 | 0.0060452223939945 |
| **Euclidean-Distance** | 0.5236792025119907 | 0.09477830320795848 | 0.8282880514581903 | 0.5806821003749957 | 0.16004503908626388 | 0.025614414536123734 |
| **City-Block** | 13.312634550989001 | 2.433179124702292 | 20.961504897753002 | 14.748577949264927 | 4.06685063640353 | 16.5392740988158 |

Für die durch das Modell intfloat/multilingual-e5-large erzeugten Embedding-Vektoren ist die Ähnlichkeit gegenüber dem Modell nvidia/llama-nemotron-embed-1b-v2 wesentlich höher. 

In [24]:
#Pearson correlation coefficient
#print(f'cosine_similarity: {np.corrcoef(embeddings_scores["cosine_similarity"], list(dataset_semRel2024_split["label"]))}')
#print(f'euclidean_distance: {np.corrcoef(embeddings_scores["euclidean_distance"], list(dataset_semRel2024_split["label"]))}')
#print(f'dot_product: {np.corrcoef(embeddings_scores["dot_product"], list(dataset_semRel2024_split["label"]))}')
#print(f'city_block: {np.corrcoef(embeddings_scores["city_block"], list(dataset_semRel2024_split["label"]))}')

In [17]:
res = stats.pearsonr(embeddings_scores['cosine_similarity'], list(dataset_semRel2024_split['label']))
print(f'cosine_similarity: {res.statistic}')
res = stats.pearsonr(embeddings_scores['euclidean_distance'], list(dataset_semRel2024_split['label']))
print(f'euclidean_distance: {res.statistic}')
res = stats.pearsonr(embeddings_scores['dot_product'], list(dataset_semRel2024_split['label']))
print(f'dot_product: {res.statistic}')
res = stats.pearsonr(embeddings_scores['city_block'], list(dataset_semRel2024_split['label']))
print(f'city_block: {res.statistic}')

cosine_similarity: 0.8290971083681944
euclidean_distance: -0.8199536208312529
dot_product: 0.82909710916587
city_block: -0.819776527021898


Modell: nvidia/llama-nemotron-embed-1b-v2

cosine_similarity: 0.8290972197420683  
euclidean_distance: -0.8199465544843246  
dot_product: 0.829097220508353  
city_block: -0.8197692466577075

Modell: intfloat/multilingual-e5-large

cosine_similarity: 0.7992037056504622  
euclidean_distance: -0.8011403937789902  
dot_product: 0.799203698347142  
city_block: -0.8004492661033433

In [18]:
#Spearman’s rank correlation coefficient
rho, p_value = stats.spearmanr(embeddings_scores['cosine_similarity'], list(dataset_semRel2024_split['label']))
print(f'cosine_similarity: {rho}')
rho, p_value = stats.spearmanr(embeddings_scores['euclidean_distance'], list(dataset_semRel2024_split['label']))
print(f'euclidean_distance: {rho}')
rho, p_value = stats.spearmanr(embeddings_scores['dot_product'], list(dataset_semRel2024_split['label']))
print(f'dot_product: {rho}')
rho, p_value = stats.spearmanr(embeddings_scores['city_block'], list(dataset_semRel2024_split['label']))
print(f'city_block: {rho}')

cosine_similarity: 0.8206032719389276
euclidean_distance: -0.8206032719389276
dot_product: 0.8206032719389276
city_block: -0.820391063336641


Modell: nvidia/llama-nemotron-embed-1b-v2

cosine_similarity: 0.8205928794532261  
euclidean_distance: -0.8205933309143886  
dot_product: 0.8205928794532261  
city_block: -0.8203707892471589

Modell: intfloat/multilingual-e5-large

cosine_similarity: 0.7834689603258412  
euclidean_distance: -0.7834689603258412  
dot_product: 0.7834689603258412  
city_block: -0.7829193937841402

Modell: nvidia/llama-nemotron-embed-1b-v2

| Score | Pearson | Spearman’s rank |
| - | - | - |
| **cosine_similarity** | 0.8290972197420683 | 0.8205928794532261 |
| **dot_product** | 0.829097220508353 | 0.8205928794532261 |
| **euclidean_distance** | -0.8199465544843246 | -0.8205933309143886 |
| **city_block** | -0.8197692466577075 | -0.8203707892471589 |

Aufgrund der bereits normierten Embedding-Vektoren führt die Berechnung der Ähnlichkeiten mit Hilfe der Cosine-Similarity und des Dot-Products sowie der daraus resultierenden Korrelationskoeffizienten zu einem nahezu gleichen Ergebnis. Gegenüber den Referenz-Scores besteht ein starker positiver Zusammenhang und die berechneten Ähnlichkeiten richten sich mit einem Werte von ca. 0.829 für den Pearson correlation coefficient und 0.821 für den Spearman’s rank correlation coefficient (Tabelle ) auch hier gut an den durch die Annotatoren bewerteten Ähnlichkeiten der Satz-Paare aus.

Die berechneten Ähnlichkeiten für die Embedding-Vektoren per Euclidean-Distance und City-Block sowie deren Korrelationskoeffizienten führen ebenso zu einem sehr ähnlichen Ergebnis, da sich die Berechnung der Entfernungen nur durch die Verwendung der Absolut-Werte (City-Block) und Quadrat-Werte (Euclidean-Distance) unterscheidet. Bezogen auf die Referenz-Scores, besteht wie erwartet mit einem Wert von ca. -0.82 für den Pearson und Spearman’s rank correlation coefficient ein negativer Zusammenhang, da geringere Entfernungen eine höhere Ähnlichkeit bedeuten, die Werte dabei also gegen 0 fallen, und für die Referenz-Scores bei höherer Ähnlichkeit gegen 1 ansteigen.

Modell: intfloat/multilingual-e5-large

| Score | Pearson | Spearman’s rank |
| - | - | - |
| **cosine_similarity** | 0.7992037056504622 | 0.7834689603258412 |
| **dot_product** | 0.799203698347142 | 0.7834689603258412 |
| **euclidean_distance** | -0.8011403937789902 | -0.7834689603258412 |
| **city_block** | -0.8004492661033433 | -0.7829193937841402 |

Verglichen mit dem Modell nvidia/llama-nemotron-embed-1b-v2 ist die Ausrichtung der berechneten Ähnlichkeiten der Embedding-Vektoren des Modells intfloat/multilingual-e5-large an den Referenz-Scores etwas geringer.

## Test

Test zur Prüfung des Verhaltens der Methoden/Modelle unter verschiedenen Bedingungen wie ähnlicher und gegensätzlicher Formulierungen und unterschiedlicher Textlängen.

Generierung entsprechender Paare von Aussagen und Berechnung der Ähnlichkeit der Embedding-Vektoren für 4 Szenarien.

1. Ähnliche Aussagen+Wortanzahl
2. Ähnliche Aussagen mit unterschiedlicher Wortanzahl
3. Gegensätzliche Aussage
4. Gegensätzliche Aussage mit gemeinsamen Wörtern

Aufgrund der geringen Anzahl an Beispielen ist der Test nicht repräsentativ.

In [19]:
# Mock Test
# Test der Berechnung der Ähnlichkeit
# Beispiel-Aussage sowie ähnliche und gegensätzliche Formulierung durch LLM generieren

# Ähnliche Aussagen
#
#Erstellung einer Aussage und einer alternativen Formulierung der selben Aussage
#
#Prompt:
#Create a statement consisting of 2–3 sentences and output it alongside an alternative phrasing of the same statement. The sequence of sentences in the alternative phrasing should not follow the same order as the sentences in the original statement.
#
#Modell: Gemini 3 3.5 Flash
#
#Original Statement:
#Studying a new language opens doors to understanding different cultures more deeply. It also sharpens your cognitive skills and improves memory over time. For these reasons, language learning is a highly rewarding pursuit.
#
#Alternative Phrasing:
#Language learning is a highly rewarding pursuit for a couple of key reasons. Over time, it sharpens your cognitive skills and improves memory. Additionally, studying a new language opens doors to understanding different cultures more deeply.

statement_original = 'Studying a new language opens doors to understanding different cultures more deeply. It also sharpens your cognitive skills and improves memory over time. For these reasons, language learning is a highly rewarding pursuit.'
statement_similiar = 'Language learning is a highly rewarding pursuit for a couple of key reasons. Over time, it sharpens your cognitive skills and improves memory. Additionally, studying a new language opens doors to understanding different cultures more deeply.'

statement_embeddings = openai_client_embedding.embeddings.create(
    input=[statement_original, statement_similiar],
    model=model_embedding
)

In [20]:
# Berechnung der Ähnlichkeit zwischen Original- und ähnlicher Aussage
print(f'cosine_similarity: {cosine_similarity(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'euclidean_distance: {euclidean_distance(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'dot_product: {dot_product(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'city_block: {city_block(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')

cosine_similarity: 0.982405050418045
euclidean_distance: 0.1875897128340505
dot_product: 0.9824050838419354
city_block: 6.726570531347534


Modell: nvidia/llama-nemotron-embed-1b-v2

cosine_similarity: 0.982390585778623  
euclidean_distance: 0.18766680510108016  
dot_product: 0.9823906218758216  
city_block: 6.724912135241539

Modell: intfloat/multilingual-e5-large

cosine_similarity: 0.9809145385194109  
euclidean_distance: 0.1953738059273161  
dot_product: 0.980914566307762  
city_block: 4.961812087891303

In [21]:
# Ähnliche Aussagen mit unterschiedlicher Anzahl Wörter
#
#Erstellung einer Aussage und einer alternativen Formulierung der selben Aussage mit unterschiedlicher Anazhl an Wörtern
#
#Prompt:
#Create a statement consisting of 2–3 sentences and output it alongside an alternative phrasing of the same statement. The two statements should contain a different number of words.
#
#Modell: Gemini 3 3.5 Flash
#
#Original Statement:
#Learning a new language can be a challenging but deeply rewarding experience. It opens up doors to new cultures and allows you to connect with people from all over the world. Dedicated practice is the ultimate key to achieving fluency.
#
#Alternative Phrasing:
#Mastering a foreign language is difficult yet incredibly fulfilling because it unveils diverse cultures and connects you globally. Consistent practice is essential for fluency.

#Modell: GPT-5.5 Instant
#
#Statement:
#The project is progressing steadily, and the team expects to finish on schedule. Everyone is reviewing the final details carefully.
#
#Alternative phrasing:
#Work on the project continues at a consistent pace, with completion still expected by the planned deadline, while every team member carefully checks the remaining details.

statement_similiar_word_count_different_1 = 'Learning a new language can be a challenging but deeply rewarding experience. It opens up doors to new cultures and allows you to connect with people from all over the world. Dedicated practice is the ultimate key to achieving fluency.'
statement_similiar_word_count_different_2 = 'Mastering a foreign language is difficult yet incredibly fulfilling because it unveils diverse cultures and connects you globally. Consistent practice is essential for fluency.'

#statement_similiar_word_count_different_1 = 'The project is progressing steadily, and the team expects to finish on schedule. Everyone is reviewing the final details carefully.'
#statement_similiar_word_count_different_2 = 'Work on the project continues at a consistent pace, with completion still expected by the planned deadline, while every team member carefully checks the remaining details.'

statement_embeddings = openai_client_embedding.embeddings.create(
    input=[statement_similiar_word_count_different_1, statement_similiar_word_count_different_2],
    model=model_embedding
)

In [22]:
print(f'cosine_similarity: {cosine_similarity(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'euclidean_distance: {euclidean_distance(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'dot_product: {dot_product(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'city_block: {city_block(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')

cosine_similarity: 0.7944656293854011
euclidean_distance: 0.641146438514582
dot_product: 0.7944656571981275
city_block: 23.06125133912201


Modell: nvidia/llama-nemotron-embed-1b-v2

cosine_similarity: 0.794315980652068  
euclidean_distance: 0.6413797950784332  
dot_product: 0.7943159861338992  
city_block: 23.065805628654743

Modell: intfloat/multilingual-e5-large

cosine_similarity: 0.9237514611364647  
euclidean_distance: 0.39050873189720614  
dot_product: 0.9237514124395491  
city_block: 10.007542812085376

In [23]:
# Gegensätzliche Aussagen
#
# Erstellung einer Aussage und einer gegensätzlichen Aussage
#
#Prompt:
#Create a statement consisting of 2–3 sentences and output it alongside a statement that is either contradictory or thematically different.
#
#Modell: Gemini 3 3.5 Flash
#
#Statement 1:
#Exploring deep space requires advanced propulsion technologies that can withstand extreme cosmic environments. Over the next few decades, scientists hope to launch autonomous probes to our neighboring star systems. These missions could fundamentally reshape our understanding of the universe.
#
#Statement 2 (Thematically Different):
#Baking the perfect loaf of sourdough bread relies heavily on maintaining a healthy, active wild yeast starter. Simple variables like ambient room temperature and water quality can drastically alter the final texture and flavor of the crust.

statement_original = 'Exploring deep space requires advanced propulsion technologies that can withstand extreme cosmic environments. Over the next few decades, scientists hope to launch autonomous probes to our neighboring star systems. These missions could fundamentally reshape our understanding of the universe.'
statement_different = 'Baking the perfect loaf of sourdough bread relies heavily on maintaining a healthy, active wild yeast starter. Simple variables like ambient room temperature and water quality can drastically alter the final texture and flavor of the crust.'

statement_embeddings = openai_client_embedding.embeddings.create(
    input=[statement_original, statement_different],
    model=model_embedding
)

In [24]:
# Berechnung der Ähnlichkeit zwischen Original- und gegensätzlicher Aussage
print(f'cosine_similarity: {cosine_similarity(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'euclidean_distance: {euclidean_distance(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'dot_product: {dot_product(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'city_block: {city_block(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')

cosine_similarity: 0.09512896728186568
euclidean_distance: 1.3452665003995588
dot_product: 0.0951289615870334
city_block: 48.103450065827076


Modell: nvidia/llama-nemotron-embed-1b-v2

cosine_similarity: 0.09547255287650623  
euclidean_distance: 1.3450110924911587  
dot_product: 0.09547254995685904  
city_block: 48.0875882662549

Modell: intfloat/multilingual-e5-large

cosine_similarity: 0.7736715674373181  
euclidean_distance: 0.67279781412841  
dot_product: 0.7736716248154833  
city_block: 17.190524476101018

In [25]:
# Gegensätzliche Aussage mit gemeinsamen Worten
#
#Zwei gegensätzliche Aussagen sollen mehrere gemeinsame Worte teilen
#
#Prompt:
#Create a statement consisting of 2–3 sentences and output it alongside a statement on a different topic. Both statements should share several words.
#
#Modell: Gemini 3 3.5 Flash
#
#Statement 1:
#The bright stars in the night sky always inspire wonder. Astronomers study these distant worlds to learn how the universe was formed.
#
#Statement 2:
#Dedicated teachers in the school inspire students every day. Young minds study hard night and day under a bright future.

statement_different_shared_words_1 = 'The bright stars in the night sky always inspire wonder. Astronomers study these distant worlds to learn how the universe was formed.'
statement_different_shared_words_2 = 'Dedicated teachers in the school inspire students every day. Young minds study hard night and day under a bright future.'

statement_embeddings = openai_client_embedding.embeddings.create(
    input=[statement_different_shared_words_1, statement_different_shared_words_2],
    model=model_embedding
)

In [26]:
# Gegensätzliche Aussage mit gemeinsamen Worten
#
#Zwei gegensätzliche Aussagen sollen möglichst viele Worte teilen
#
#Prompt:
#Create a statement consisting of 2–3 sentences and output it alongside a statement on a different topic. Both statements should share as many words as possible.
#
#Modell: Gemini 3 3.5 Flash
#
#Statement 1:
#The deep space telescope captured a brilliant new star shining brightly in the distant galaxy. This remarkable discovery completely changed how the brilliant scientists viewed the universe.
#
#Statement 1:
#The deep ocean resort captured a brilliant new market shining brightly in the distant region. This remarkable discovery completely changed how the brilliant directors viewed the luxury industry.

statement_different_shared_words_1 = 'The deep space telescope captured a brilliant new star shining brightly in the distant galaxy. This remarkable discovery completely changed how the brilliant scientists viewed the universe.'
statement_different_shared_words_2 = 'The deep ocean resort captured a brilliant new market shining brightly in the distant region. This remarkable discovery completely changed how the brilliant directors viewed the luxury industry.'

statement_embeddings = openai_client_embedding.embeddings.create(
    input=[statement_different_shared_words_1, statement_different_shared_words_2],
    model=model_embedding
)

In [27]:
print(f'cosine_similarity: {cosine_similarity(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'euclidean_distance: {euclidean_distance(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'dot_product: {dot_product(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')
print(f'city_block: {city_block(statement_embeddings.data[0].embedding, statement_embeddings.data[1].embedding)}')

cosine_similarity: 0.5164880391996843
euclidean_distance: 0.9833737313733047
dot_product: 0.5164880252887223
city_block: 35.218497616863715


Modell: nvidia/llama-nemotron-embed-1b-v2

cosine_similarity: 0.5164880391996843  
euclidean_distance: 0.9833737313733047  
dot_product: 0.5164880252887223  
city_block: 35.218497616863715

Modell: intfloat/multilingual-e5-large

cosine_similarity: 0.8649199749792519  
euclidean_distance: 0.5197692265717985  
dot_product: 0.8649199712960702  
city_block: 13.266035968318647

nvidia/llama-nemotron-embed-1b-v2

| Eingabe | Cosine-Similarity | Dot-Product | Euclidean-Distance | City-Block |
| - | - | - | - | - |
| Ähnliche Aussagen+Wortanzahl | 0.982390585778623 | 0.9823906218758216 | 0.18766680510108016 | 6.724912135241539 |
| Ähnliche Aussagen mit unterschiedlicher Wortanzahl | 0.794315980652068 | 0.7943159861338992 | 0.6413797950784332 | 23.065805628654743 |
| Gegensätzliche Aussagen | 0.09547255287650623 | 0.09547254995685904 | 1.3450110924911587 | 48.0875882662549 |
| Gegensätzliche Aussagen mit gemeinsamen Wörtern | 0.5164880391996843 | 0.5164880252887223 | 0.9833737313733047 | 35.218497616863715 |

Die durch das LLM erzeugten ähnlichen Aussagen mit ähnlicher Wortanzahl werden durch das Modell nvidia/llama-nemotron-embed-1b-v2 sehr gut erkannt und mittels Cosine-Similarity und Dot-Product wurde eine Ähnlichkeit von ca. 0.982 berechnet wohingegen die gemessene Ähnlichkeit bei unterschiedlicher Wortanzahl mit einem Wert von ca. 0.794 deutlich geringer ist (Tabelle ). Für die gegensätzlichen Aussagen wurde ebenso eine mit einem Wert von ca. 0.095 entsprechend geringe Ähnlichkeit berechnet, teilen sich die Aussagen gemeinsame Wörter, fällt diese mit ca. 0.516 demgegenüber jedoch eher hoch aus.

intfloat/multilingual-e5-large

| Eingabe | Cosine-Similarity | Dot-Product | Euclidean-Distance | City-Block |
| - | - | - | - | - |
| Ähnliche Aussagen+Wortanzahl | 0.9809145385194109 | 0.980914566307762 | 0.1953738059273161 | 4.961812087891303 |
| Ähnliche Aussagen mit unterschiedlicher Wortanzahl | 0.9237514611364647 | 0.9237514124395491 | 0.39050873189720614 | 10.007542812085376 |
| Gegensätzliche Aussagen | 0.7736715674373181 |  0.7736716248154833 | 0.67279781412841 | 17.190524476101018 |
| Gegensätzliche Aussagen mit gemeinsamen Wörtern | 0.8649199749792519 | 0.8649199712960702 | 0.5197692265717985 | 13.266035968318647 |

Die Nutzung des Modells intfloat/multilingual-e5-large führt sowohl für die ähnlichen Aussagen mit ähnlicher wie auch unterschiedlicher Wortanzahl zu einer sehr hohen Ähnlichkeit von ca. 0.981 bzw. 0.924 unter Verwendung der Cosine-Similarty und des Dot-Products (Tabelle ). Verglichen mit dem Modell nvidia/llama-nemotron-embed-1b-v2 wurde für die gegensätzlichen Aussagen mit einem Wert von ca. 0.774 eine sehr hohe Ähnlichkeit gemessen, welche beim Vorhandensein gemeinsamer Wörter auf 0.865 ansteigt.

Bei Betrachtung der ähnlichen Aussagen mit ähnlicher und unterschiedlicher Wortanzahl, steigt für beide Modelle die berechnet Ähnlichkeit mit Hilfe von Euclidean-Distance/City-Block und Werten von ca. 0.188/6.725 (gleiche Wortzahl) und 0.641/23.1 (unterschiedliche Wortzahl) für das Modell nvidia/llama-nemotron-embed-1b-v2 sowie 0.195/4.962 und 0.391/10.0 für das Modell intfloat/multilingual-e5-large gegenüber der Veränderung der mit Cosine-Similarity/Dot-Product berechneten Werte (0.982 und 0.794 nvidia/llama-nemotron-embed-1b-v2, 0.981 und 0.924 intfloat/multilingual-e5-large) um den Faktor 2 bzw. 3, was auf die unterschiedlichen Textlängen zurückzuführen ist.

## Auswahl Modell+Methode

Bezogen auf die ähnlichen Aussagen mit ähnlicher und unterschiedlicher Wortzahl ist die berechnete Ähnlichkeit für beide Modelle 

Im Vergleich ist die Ähnlichkeit der ähnlichen Aussagen mit gleicher und unterschiedlicher Anzahl an Wörtern für beide Modelle mit 0.794-0.982 relativ hoch, wobei der Wert für das Modell intfloat/multilingual-e5-large bei unterschiedlicher Anzahl an Wörtern gegenüber dem Modell nvidia/llama-nemotron-embed-1b-v2 entschieden höher ist.

Für die gegensätzlichen Aussagen schneidet das Modell nvidia/llama-nemotron-embed-1b-v2 wesentlich besser ab und die Ähnlichkeit ist, der Eingabe entsprechend, mit einem Wert von 0.095 sehr gering. Die Verwendung des Modells intfloat/multilingual-e5-large führt für diese Aussage hingegen zu einem sehr hohen Ähnlichkeitswerte von 0.774.

Aufgrund der Größe des Kontext-Fensters von 8192 Token, welches die Verarbeitung größerer Text-Einheiten ermöglicht und der stärkeren Komprimierung der Eingabe auf eine Dimension von 2048, verbunden mit der Möglichkeit der Kürzung, sowie der erzielten Ergebnisse, wird nvidia/llama-nemotron-embed-1b-v2 als Modell für die Generierung der Text-Embeddings ausgewählt und in den Experimenten verwendet.

Die Wahl der Methode zur Berechung der Ähnlichkeit hängt vom verwendeten Embedding-Modell und der Normierung der Vektoren ab. In der Praxis der natürlichen Sprachverarbeitung ist die Cosine-Similarity der Standard. Da die durch das Embedding-Modell erzeugten Vektoren bereits normiert sind und aufgrund des geringeren Rechenaufwandes, verglichen mit der Cosine-Similarity, wird für die Berechnung der Ähnlichkeiten das Dot Product als Methode verwendet.